# Boxplot: número de palabras por personaje, por sexo

Notebook adaptado del script `word_count_boxplot.py`. Ya incluye todas las funciones necesarias (no depende de `data_prep.py`).

Recuerda ajustar `RAW_PATH` a la ubicación de tu archivo `Dataset_final.pkl`.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# En Jupyter/JupyterLab, si fig.show() no te muestra nada, descomenta la línea
# que corresponda a tu entorno:
# pio.renderers.default = "notebook"   # Jupyter Notebook clásico
# pio.renderers.default = "iframe"     # JupyterLab
# pio.renderers.default = "colab"      # Google Colab

In [2]:
# --- Ruta al dataset: cámbiala por la ruta local donde tengas el .pkl ---
RAW_PATH = "Dataset_final.pkl"

BG_COLOR = "#0f1117"
GRID_COLOR = "rgba(255,255,255,0.08)"
TEXT_COLOR = "#e8e8e8"
COLOR_MAP = {"female": "#e0a72e", "male": "#a3a3a3"}
LABEL_MAP = {"female": "Femenino", "male": "Masculino"}

## 1. Carga y preparación de datos

In [3]:
import pandas as pd


def build_character_long_df(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        script = row["Script_Dict"] or {}
        genders = row["Characters_Genders"] or {}
        for character, text in script.items():
            if not text or not text.strip():
                continue
            gender = genders.get(character, "unknown")
            rows.append({
                "IMDb_ID": row["IMDb_ID"], "Title": row["Title"], "Award": row["Award"],
                "Character": character, "Gender": gender, "Text": text,
            })
    return pd.DataFrame(rows)

def load_all():
    df = pd.read_pickle(RAW_PATH)   # <- sin read_parquet, sin json.loads

    df_unique_movies = df.drop_duplicates(subset="IMDb_ID", keep="first")
    long_unique = build_character_long_df(df_unique_movies)
    long_by_award = build_character_long_df(df)

    long_unique = long_unique[long_unique["Gender"].isin(["male", "female"])].copy()
    long_by_award = long_by_award[long_by_award["Gender"].isin(["male", "female"])].copy()

    return long_unique, long_by_award

long_unique, long_by_award = load_all()
print(long_unique.shape)
print(long_unique.head())

(2310, 6)
     IMDb_ID                                          Title         Award  \
0  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
1  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
2  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
3  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
4  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   

  Character  Gender                                               Text  
0   ARAGORN    male  No news of Frodo? We still have time. What doe...  
1     ARWEN  female  What did you see? You have the gift of foresig...  
2     BILBO    male  Tell me again, lad. Where are we going? Frodo ...  
3  DENETHOR    male  Perhaps you come to explain this? Perhaps you ...  
4    DÉAGOL    male  Sméagol, I've got one!I've got a fish, Sméagol...  


## 2. Conteo de palabras por personaje

In [4]:
def add_word_count(long_df: pd.DataFrame) -> pd.DataFrame:
    long_df = long_df.copy()
    long_df["Word_Count"] = long_df["Text"].str.split().str.len()
    return long_df


long_unique = add_word_count(long_unique)
long_unique.groupby("Gender")["Word_Count"].describe()

,count,mean,std,min,25%,50%,75%,max
Gender,,,,,,,,
female,687.0,245.947598,566.278056,1.0,12.0,48.0,214.5,5523.0
male,1623.0,253.727049,634.165298,1.0,12.0,41.0,171.0,6638.0


## 3. Función del boxplot

In [5]:
def make_boxplot_fig(long_df: pd.DataFrame, title: str, log_y: bool = False):
    fig = go.Figure()
    for gender in ["male", "female"]:
        sub = long_df[long_df["Gender"] == gender]
        fig.add_trace(
            go.Box(
                y=sub["Word_Count"],
                name=LABEL_MAP[gender],
                marker_color=COLOR_MAP[gender],
                line=dict(color=COLOR_MAP[gender]),
                boxmean=True,  # muestra también la media, no solo la mediana
                boxpoints="outliers",
            )
        )

    fig.update_layout(
        title=dict(text=title, font=dict(size=18, color=TEXT_COLOR)),
        plot_bgcolor=BG_COLOR,
        paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR, family="Arial"),
        yaxis=dict(
            title="Nº de palabras por personaje" + (" (escala log)" if log_y else ""),
            gridcolor=GRID_COLOR,
            type="log" if log_y else "linear",
        ),
        xaxis=dict(title="Género del personaje"),
        showlegend=False,
        margin=dict(l=60, r=40, t=60, b=50),
    )
    return fig

## 4. Gráficos

En un notebook, basta con dejar la figura como última expresión de la celda (o usar `fig.show()`) para que se renderice inline.

In [6]:
# Escala lineal (se ven mejor los outliers extremos)
fig_linear = make_boxplot_fig(
    long_unique, "Número de palabras por personaje, por sexo", log_y=False
)
fig_linear.show()

In [7]:
# Escala logarítmica (recomendable: la distribución está muy sesgada,
# con personajes de 1 palabra y otros de +6000, la escala log deja ver
# mejor la caja central)
fig_log = make_boxplot_fig(
    long_unique, "Número de palabras por personaje, por sexo (escala log)", log_y=True
)
fig_log.show()